# 25 Western Shimane 2018 — Colab cells N1–N18

Runtime: CPU, high RAM (do not change it during the run). Run the cells one by one in the order of 25_STEPS_JA.md (do not use Run all). Part 0 (push and Zenodo) must be finished before N5. NR is only for a reconnection (see the last cell).

## N1 — Drive と道具
- 期待: `seisbench 0.12.7`(24_07 の dev と同じ版)。表示の 1 行を控える(事前登録 14 節の 7)。
- pip の後に「セッションを再起動」と出たら、再起動して N1 からやり直す。

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!pip -q install HinetPy==0.13.0 seisbench==0.12.7 obspy
import h5py, torch, numpy, scipy, obspy, seisbench; print(h5py.__version__, torch.__version__, numpy.__version__, scipy.__version__, 'obspy', obspy.__version__, 'seisbench', seisbench.__version__)

## N2 — 場所の変数と、スクリプト・震源ファイルの SHA-256
- 期待: 8 行とも `OK`。最後の行が `[]`。`NG` が 1 つでもあれば止める。

In [ ]:
import hashlib, os
SRC = '/content/drive/MyDrive/ARGUS/src'
CAT = '/content/drive/MyDrive/ARGUS/data/catalog'
DRV = '/content/drive/MyDrive/ARGUS_shimane2018'
LO  = '/content/ARGUS_shimane2018'
R1  = 'data/tottori_processed/tottori_parsed_realpick_shimane2018_m15_r1.pt'
HYPO = f'{CAT}/jma_raw/h2018_25_00.zip'
REF_H5 = '/content/drive/MyDrive/2026WTottori/data/waves/tottori_all_100Hz.h5'
REF_CACHE_2026 = '/content/drive/MyDrive/ARGUS_tottori/data/tottori_processed/tottori_parsed_realpick_m15.pt'
EXP = {  # 事前登録の記録の 9 節と同じ値
    '25_01_shimane2018_jma_catalog.py': 'a147484fea5472cbb2364052d9a153d22425c7ec10630cc729ef223280240ce7',
    '25_02_shimane2018_hinet_to_h5.py': '220d3c72e0bb4631d1f180571e0a7e177701662a406a3eef4b3c1a87e615c519',
    '25_03_shimane2018_realpick_cache_colab.py': 'd58b389f4f64e4b068b75f7b6a9ee41d23a241ec0579a90cbf81ffe0d53363f4',
    '25_03h_shimane2018_overlap_exclusion.py': 'e676ea476c098c68ee6461f21d40df1541a799c6497994d85a32df26620cebaa',
    '25_04d_shimane2018_gclock2_picks.py': '419a09119d38e89a04c7504cefd361aef1ab7a531fda95b8f77501d9dad50b4f',
    '25_04e_shimane2018_station_pick_diagnose.py': '25baefde700ff9d867a518d6bd306afd2b1a8c97ae6fcadd25120133ec629d06',
    '24_07_causal_picks.py': 'b9b2fd5c8855c52f2e3a528e695374af752db3b0de6f789942e6da8824c2d14b',
}
for n, h in EXP.items():
    got = hashlib.sha256(open(f'{SRC}/{n}', 'rb').read()).hexdigest() if os.path.exists(f'{SRC}/{n}') else 'MISSING'
    print('OK  ' if got == h else 'NG  ', n, got)
hh = hashlib.sha256(open(HYPO, 'rb').read()).hexdigest() if os.path.exists(HYPO) else 'MISSING'
print('OK  ' if hh.startswith('ea8eb186db3f') else 'NG  ', 'h2018_25_00.zip', hh[:12])
print('名前に (1) などが付いた写し:', [f for f in os.listdir(SRC) if f.startswith(('25_', '24_07')) and '(' in f])

## N3 — win32tools(catwin32・win2sac_32)
- このノートの N3 のセルを回す(中身は大阪・長野のノートのセル 3 と同じ。`マイドライブ/KRITES/bin/` に作ったものを写すだけ)。
- 期待: `which catwin32 win2sac_32` に 2 つのパスが出て、`file` に `ELF` と出る。`Mach-O` と出たら Mac で作ったものなので、`KRITES/bin` の
  2 つを消して、もう一度このセルを回す。

In [ ]:
import os, glob, shutil, subprocess
KRI  = '/content/drive/MyDrive/KRITES'
KBIN = f'{KRI}/bin'
TGZ  = f'{KRI}/win32tools.tar.gz'
NEED = ['catwin32', 'win2sac_32']
if not all(os.path.isfile(f'{KBIN}/{n}') for n in NEED):
    if not os.path.isfile(TGZ):
        raise SystemExit(f'STOP: {TGZ} が無い。win32tools.tar.gz を マイドライブ/KRITES に置いてから回し直す')
    B = '/content/w32build'
    shutil.rmtree(B, ignore_errors=True); os.makedirs(B)
    subprocess.run(['tar', '-xzf', TGZ, '-C', B], check=True)
    top = [d for d in glob.glob(f'{B}/*') if os.path.isdir(d)]
    if len(top) != 1:
        raise SystemExit(f'STOP: 展開した中身が想定(フォルダ 1 つ)と違う: {top}')
    r = subprocess.run(['make'], cwd=top[0], capture_output=True, text=True)
    print(r.stdout[-3000:]); print(r.stderr[-3000:])
    if r.returncode != 0:
        raise SystemExit('STOP: make が失敗した(上の表示を貼る)')
    os.makedirs(KBIN, exist_ok=True)
    for n in NEED:
        hit = [p for p in glob.glob(f'{top[0]}/**/{n}', recursive=True) if os.path.isfile(p) and os.access(p, os.X_OK)]
        if len(hit) != 1:
            raise SystemExit(f'STOP: 作った {n} が 1 つに決まらない: {hit}')
        shutil.copy2(hit[0], f'{KBIN}/{n}')
for n in NEED:
    shutil.copy2(f'{KBIN}/{n}', f'/usr/local/bin/{n}')
    os.chmod(f'/usr/local/bin/{n}', 0o755)
!which catwin32 win2sac_32
!file /usr/local/bin/catwin32 /usr/local/bin/win2sac_32

## N4 — Hi-net の資格情報(ノートに書き残さない)


In [ ]:
import os, getpass
os.environ['HINET_USER'] = getpass.getpass('Hi-net user: ')
os.environ['HINET_PASS'] = getpass.getpass('Hi-net password: ')

## N5 — 25_01: 気象庁の震源(h2018)。**0 部が全て済んでから**
- 期待: `震源ファイル h2018_25_00.zip(SHA-256 ea8eb186db3f…`、
  `[25_01] 本震: 2018-04-09 01:32:30.810000 JST  35.1847N 132.5867E  深さ 12.13 km  M6.1D  精度 1  (想定の 1 分 01:32:00〜01:33:00 JST の中か: True。要確認の項目)`、
  `箱 [35.0347, 35.3347, 132.4117, 132.7617]`、`数:` の `all` 3361・`mj_ge_1p5` 389・`test_candidates_1p5_to_2p0_after_72h` 41(どれも 25_00 と同じ)、
  `試験の候補 41 ≥ 30: 次へ進める`。
- 違う値が出たら止めて出力を貼る。
- h2018 の確定の段階(事前登録 14 節の 4)を、気象庁の震源ファイルのページで確かめて `results/25_registration.txt` に書く。
- 出力: `{CAT}/jma_shimane2018.csv`・`{CAT}/25_01_shimane2018_catalog.json`・`/content/drive/MyDrive/ARGUS/PDF/25_01_shimane2018_catalog.pdf`

In [ ]:
assert input('0 部(push・Zenodo の新しい版・登録の控え)は全て済んだか? 済んだら yes: ').strip() == 'yes', '0 部を先に'
!python {SRC}/25_01_shimane2018_jma_catalog.py --hypo {HYPO}; echo "exit code: $?"

## N6 — 25_02: 取る計画だけ
- 確かめること: `参照の z-score の確認 … PASS`、候補局 60、区切りの数と分の合計、置き場所の見込み。

In [ ]:
!python {SRC}/25_02_shimane2018_hinet_to_h5.py --ref-h5 "{REF_H5}" --dry-run; echo "exit code: $?"

## N7 — 煙試験(本震の区間に掛かる区切りを先に 3 つ)


In [ ]:
!python {SRC}/25_02_shimane2018_hinet_to_h5.py --ref-h5 "{REF_H5}" --max-blocks 3; echo "exit code: $?"

## N7b — 煙試験の分だけで組み立ててゲートを見る
- G-ref・G-dup が PASS。`PDF/25_02_mainshock_partial.pdf` で本震の P が予想の P の印にそろっているか。G-clock の FAIL は記録だけ。
- 煙試験の G-cover は区切りが 3 つだけなので低くてよい(本番の判定は N9)。G-ref・G-dup の FAIL は止めて貼る。

In [ ]:
!python {SRC}/25_02_shimane2018_hinet_to_h5.py --ref-h5 "{REF_H5}" --assemble --partial; echo "exit code: $?"

## N8 — 全ての区切りを取る(止まっても同じセルで続きから)
- 最後の行が `終わり: 残りの区切り 0/N`(全部済んだ後に回すと `取る区切りが無い(全部済み)`)になるまで回し直す。取れた割合が 0.98 以上で、何度回しても取れない区切りだけが残った時だけ、N9 に `--allow-missing`。

In [ ]:
!python {SRC}/25_02_shimane2018_hinet_to_h5.py --ref-h5 "{REF_H5}"; echo "exit code: $?"

## N8b — Colab が切れた直後だけ


In [ ]:
!python {SRC}/25_02_shimane2018_hinet_to_h5.py --ref-h5 "{REF_H5}" --restore-selection --break-lock; echo "exit code: $?"

## N9 — 組み立てとゲート(Colab のローカルで)
- `STOP(G-clock だけ)` → 進む(`shimane2018_100Hz_FAILED.h5` と `stations_shimane2018.csv.part` を N10a が自分で選ぶ)。それ以外の STOP は止めて出力を貼る。
- G-cover が 0.80〜0.95 の「注意」は進む(止めるのは 0.80 未満だけ。事前登録 5.2 節)。
- 報告の `version` が `v1 (=24_02 v1 = 21_20b v1 + window scale)`。`df -h` の空きが、N6 の見込み(区切り + h5)より大きいこと。

In [ ]:
!df -h /content
!mkdir -p {LO}/data
!cp -r {DRV}/data/blocks {LO}/data/ && cp {DRV}/data/25_02_plan.json {LO}/data/
# N8 で取れない区切りだけが残った時(取れた割合 0.98 以上)だけ、下の行の --out {LO} の後に --allow-missing を付ける
!python {SRC}/25_02_shimane2018_hinet_to_h5.py --ref-h5 "{REF_H5}" --assemble --out {LO}; echo "exit code: $?"
!mkdir -p {DRV}/data/catalog {DRV}/PDF && cp {LO}/data/25_02_report.json {DRV}/data/ && cp {LO}/PDF/25_02_*.pdf {DRV}/PDF/
!cp {LO}/data/catalog/stations_shimane2018.csv* {DRV}/data/catalog/
!ls -la {LO}/data/waves {LO}/data/catalog

## N10a — 変数だけ(h5・観測点・箱)。Colab が切れた後もこのセルだけを回し直す
- 期待: `box 35.0347 35.3347 132.4117 132.7617`、2 行とも `True`。

In [ ]:
import json, os
S = json.load(open(f'{CAT}/25_01_shimane2018_catalog.json')); BOX = ' '.join(str(x) for x in S['box']); print('box', BOX)
H5 = f'{LO}/data/waves/shimane2018_100Hz.h5'
STA = f'{LO}/data/catalog/stations_shimane2018.csv'
if not os.path.exists(H5):            # G-clock だけの FAIL の時(事前登録 5.2 節)
    H5, STA = f'{LO}/data/waves/shimane2018_100Hz_FAILED.h5', STA + '.part'
IDX = f'{LO}/data/tottori_processed/h5_index_{os.path.basename(H5)[:-3]}.npz'   # 25_03 が作る窓の索引(N15 で使う)
print(H5, os.path.exists(H5)); print(STA, os.path.exists(STA))

## N10b — 25_03 の煙試験(10 地震)


In [ ]:
!python {SRC}/25_03_shimane2018_realpick_cache_colab.py --h5 {H5} --jma {CAT}/jma_shimane2018.csv --stations {STA} --min-mag 1.5 --box {BOX} --base {LO} --tag _shimane2018_smoke --max-events 10; echo "exit code: $?"

## N11 — 25_03 の本番(1 回だけ)
- 出力: `{LO}/data/tottori_processed/tottori_parsed_realpick_shimane2018_m15.pt`・`{LO}/models/locator/15_01c_build_report_shimane2018_m15.json`
- 最後の行の値が 9 節の 25_03 の値と同じこと。

In [ ]:
!python {SRC}/25_03_shimane2018_realpick_cache_colab.py --h5 {H5} --jma {CAT}/jma_shimane2018.csv --stations {STA} --min-mag 1.5 --box {BOX} --base {LO} --tag _shimane2018_m15; echo "exit code: $?"
!python -c "import json;print('25_03 script_sha256', json.load(open('{LO}/models/locator/15_01c_build_report_shimane2018_m15.json'))['script_sha256'])"

## N11b — キャッシュをすぐ Drive に写す


In [ ]:
!mkdir -p {DRV}/data/tottori_processed {DRV}/models/locator
!cp {LO}/data/tottori_processed/tottori_parsed_realpick_shimane2018_m15.pt {DRV}/data/tottori_processed/
!cp {LO}/models/locator/15_01c_build_report_shimane2018_m15.json {DRV}/models/locator/

## N12 — 25_03h: R1(重なった地震を除く)
- 期待: `G-match PASS`、`除いた後: … 試験の候補 N(前 M)` で N ≥ 30、`出力: …_shimane2018_m15_r1.pt  … SHA-256 …`。
- `STOP(R2)`・`STOP(R3)` → 止める(R3 は INCONCLUSIVE として報告。事前登録 5.4 節)。
- 本震の不一致・G-match などその他の STOP も止めて出力を貼る。

In [ ]:
!python {SRC}/25_03h_shimane2018_overlap_exclusion.py --cache {LO}/data/tottori_processed/tottori_parsed_realpick_shimane2018_m15.pt --catalog {CAT}/jma_shimane2018.csv --report {LO}/data/25_02_report.json --out {LO}; echo "exit code: $?"

## N12b — R1 の出力をすぐ Drive に写す
- 期待: 最初の 2 行の値が同じ。この値と最後の行の値を、25_05 を開く前に `results/25_registration.txt` に書く(事前登録 6 節)。
- N12 が STOP(R2・R3 など)の時は、このセルを回さず、`!mkdir -p {DRV}/data && cp {LO}/data/25_03h_overlap_exclusion.json {DRV}/data/` だけを回して記録を残し、N18 へ。

In [ ]:
!mkdir -p {DRV}/data/tottori_processed && cp {LO}/{R1} {DRV}/data/tottori_processed/ && cp {LO}/data/25_03h_overlap_exclusion.json {DRV}/data/
!sha256sum {LO}/{R1} {DRV}/data/tottori_processed/tottori_parsed_realpick_shimane2018_m15_r1.pt
!python -c "import json;print('input_cache_sha256', json.load(open('{LO}/data/25_03h_overlap_exclusion.json'))['input_cache_sha256'])"

## N13 — 25_04d: G-clock2
- 最初の行が `selftest: 新しい (d)(…)の 7 例に合格`。
- 最後の行が `STOP(G-clock2)`(条件の FAIL)の時は、ここでは止めずに N14・N14b に進む(扱いは事前登録 5.3 節で決まっている)。
- それ以外の STOP(検測 100 未満・日/局が 0・報告が無い・selftest)は JSON が書かれないので、止めて出力を貼る。

In [ ]:
!python {SRC}/25_04d_shimane2018_gclock2_picks.py --base {LO} --cache {R1} --ref-cache {REF_CACHE_2026}; echo "exit code: $?"

## N14 — 25_04e(必ず回す)


In [ ]:
!python {SRC}/25_04e_shimane2018_station_pick_diagnose.py --base {LO} --cache {R1} --ref-cache {REF_CACHE_2026}; echo "exit code: $?"

## N14b — G-clock2 の判定に要る値を表示する(読むだけ)
- 読み方(事前登録 5.3 節): `pass_all`・`pass_day`・`pass_block`・`pass_events`・`pass_measured_frac` のどれかが False → 止める。
  (c-near) の外れが 1 局でもある、または 80 km 以内・n ≥ 20 の局が 0 → 止める。`pass_station` だけが False → 進み、決まった印を付ける。
  `gclock2_pass` が True → 進む。
- (c-far) の時に付ける印の英文・日本文は、事前登録 5.3 節の 4 にある。止める時も N14c と N18 は回す(記録を残す)。**変更の記録は足さない**(5.3 節の 6)。

In [ ]:
import json, csv
J = json.load(open(f'{LO}/data/25_04d_gclock2.json'))
for k in ['pass_all', 'pass_day', 'pass_station', 'pass_block', 'pass_events', 'pass_measured_frac', 'gclock2_pass']: print(k, J[k])
B = J['event_block_d']
print('  (d) 本震', J['mainshock_jst'], '+ 72 h より後の地震', B['n_events'], ' 窓', B['n_windows'], ' 最大', B['max_abs_s'], ' 窓', B['worst_window_start_jst'], '〜', B['worst_window_end_jst'])
for x in J['gclock_events']:
    print('  (e)', x['origin_jst'], 'Mj', x['mj'], 'found_dt_s', round(x['found_dt_s'], 3), 'n_picks', x['n_picks'], 'PASS' if x['pass_'] else 'FAIL')
rows = list(csv.DictReader(open(f'{LO}/data/25_04e_station_picks.csv')))
near = [r for r in rows if float(r['epi_from_mainshock_km']) <= 80.0 and int(r['n']) >= 20]
bad_near = [r for r in near if abs(float(r['dev_s'])) > 1.0]
bad_far = [r for r in rows if float(r['epi_from_mainshock_km']) > 80.0 and int(r['n']) >= 20 and abs(float(r['dev_s'])) > 1.0]
print('(c-near) 80 km 以内・n ≥ 20 の局', len(near), '  |dev_s| > 1.0 s の局', len(bad_near))
for r in bad_near: print('   ', r['station'], r['epi_from_mainshock_km'], 'km  n', r['n'], ' dev_s', r['dev_s'])
print('(c-far) 80 km より遠い・n ≥ 20 で |dev_s| > 1.0 s の局', len(bad_far))
for r in bad_far: print('   ', r['station'], r['epi_from_mainshock_km'], 'km  n', r['n'], ' dev_s', r['dev_s'])

## N14c — G-clock2 の出力をすぐ Drive に写す


In [ ]:
!mkdir -p {DRV}/data {DRV}/PDF && cp {LO}/data/25_04d_gclock2.json {LO}/data/25_04e_station_picks.csv {DRV}/data/ && cp {LO}/PDF/25_04*.pdf {DRV}/PDF/

## N15 — 24_07: H5 の材料(τ までの波形だけの検測。全ての地震。1 回だけ)
- 期待: `selftest: 元の値への戻し(v3)・…に全て合格`、`h5 の値: win_mean・win_std から元の値に戻して使う(v3)`、
  `成分の並び ENZ → … ['EW', 'NS', 'UD'] の順に並べ替えて渡す`、見つからない局 0(10% 以下は可)、`G-causal … PASS`、τ ごとの 4 行。
- 所要時間: CPU なので 1 地震あたり数 s〜十数 s(終わりの「検測 … s(1 地震あたり … s)」の行まで、表示が止まって見える)。
- 索引 `{IDX}` が無い(切れた後)か使えない時は、その場で作り直す(数十分かかりうる)。
- 最後の行で出力をすぐ Drive に写す(N16 の前に切れても失わないため)。
- STOP が出たら N16 に進まず、出力を貼る。24_07 は直さず回し直さない。その時は Mac の 25_05 を `--no-causal` で回す(事前登録 5.5 節、M2・M3)。
- 出力: `{LO}/data/24_07_causal_picks_shimane2018.csv`・`{LO}/models/24_07_causal_picks_shimane2018.json`・`{LO}/PDF/24_07_causal_picks_shimane2018.pdf`

In [ ]:
!python {SRC}/24_07_causal_picks.py --h5 {H5} --cache {LO}/{R1} --stations {STA} --index {IDX} --base {LO} --tag _shimane2018 --all-events --label "Western Shimane 2018"; echo "exit code: $?"
!mkdir -p {DRV}/data {DRV}/models {DRV}/PDF && cp {LO}/data/24_07_causal_picks_shimane2018.csv {DRV}/data/ && cp {LO}/models/24_07_causal_picks_shimane2018.json {DRV}/models/ && cp {LO}/PDF/24_07_*.pdf {DRV}/PDF/

## N16 — Drive へ保存と SHA-256
- この 4 つの SHA-256 を控える(`results/25_registration.txt` に書く)。最後の行が N1 の seisbench の版(0.12.7)と同じこと(事前登録 14 節の 7)。

In [ ]:
!mkdir -p {DRV}/data/tottori_processed {DRV}/models {DRV}/PDF
!cp {LO}/{R1} {DRV}/data/tottori_processed/
!cp {LO}/data/25_03h_overlap_exclusion.json {LO}/data/25_04d_gclock2.json {LO}/data/25_04e_station_picks.csv {LO}/data/24_07_causal_picks_shimane2018.csv {DRV}/data/
!cp {LO}/models/24_07_causal_picks_shimane2018.json {DRV}/models/ ; cp {LO}/PDF/*.pdf {DRV}/PDF/
!sha256sum {LO}/{R1} {LO}/data/24_07_causal_picks_shimane2018.csv {LO}/models/24_07_causal_picks_shimane2018.json {CAT}/jma_shimane2018.csv
!python -c "import json;print('24_07 seisbench', json.load(open('{LO}/models/24_07_causal_picks_shimane2018.json'))['seisbench'])"

## N17 — Drive に書けたかの確かめ
- 期待: N16 の 4 つの値と同じ。

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive')
!sha256sum {DRV}/data/tottori_processed/tottori_parsed_realpick_shimane2018_m15_r1.pt {DRV}/data/24_07_causal_picks_shimane2018.csv {DRV}/models/24_07_causal_picks_shimane2018.json {CAT}/jma_shimane2018.csv

## N18 — 記録を 1 つの zip にまとめる(Colab で止めた時も、ここまでに回したセルの分を必ず回す)
- 期待: 最後の行に zip の場所と数。Mac の M6a は、この `ARGUS_shimane2018/25_reports.zip` 1 つだけを落とす。

In [ ]:
import os, zipfile, glob
os.makedirs(DRV, exist_ok=True)       # N5 などで止めた時は、まだ無い
OUTZ = f'{DRV}/25_reports.zip'
F = [f'{CAT}/25_01_shimane2018_catalog.json', f'{CAT}/jma_shimane2018.csv', '/content/drive/MyDrive/ARGUS/PDF/25_01_shimane2018_catalog.pdf',
     f'{DRV}/data/25_02_report.json', f'{DRV}/data/25_02_plan.json', f'{DRV}/models/locator/15_01c_build_report_shimane2018_m15.json',
     f'{DRV}/data/25_03h_overlap_exclusion.json', f'{DRV}/data/25_04d_gclock2.json', f'{DRV}/data/25_04e_station_picks.csv',
     f'{DRV}/models/24_07_causal_picks_shimane2018.json'] + sorted(glob.glob(f'{DRV}/PDF/*.pdf'))
with zipfile.ZipFile(OUTZ, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in F:
        if os.path.exists(p): z.write(p, os.path.basename(p))
        else: print('無い(そこまで回していない時は正常):', os.path.basename(p))
print(OUTZ, len(zipfile.ZipFile(OUTZ).namelist()), '個')

## NR — Drive から戻す(Colab が切れた時だけ。下の表で指示された時に回す)
- 期待: `_r1.pt` があれば、その値が N12b の値と同じ。

In [ ]:
!mkdir -p {LO}/data/tottori_processed {LO}/models/locator {LO}/PDF
!cp {DRV}/data/tottori_processed/tottori_parsed_realpick_shimane2018_m15.pt {LO}/data/tottori_processed/
!cp {DRV}/models/locator/15_01c_build_report_shimane2018_m15.json {LO}/models/locator/
!cp {DRV}/data/25_02_report.json {LO}/data/
!cp {DRV}/data/tottori_processed/tottori_parsed_realpick_shimane2018_m15_r1.pt {LO}/data/tottori_processed/ 2>/dev/null; cp {DRV}/data/25_03h_overlap_exclusion.json {LO}/data/ 2>/dev/null
!cp {DRV}/data/25_04d_gclock2.json {DRV}/data/25_04e_station_picks.csv {LO}/data/ 2>/dev/null; cp {DRV}/PDF/25_04*.pdf {LO}/PDF/ 2>/dev/null
!sha256sum {LO}/data/tottori_processed/*.pt

## Colab が切れた時
まず N1 → N2 → N3 → N4。その後:

| 切れた所 | 次に回すもの |
|---|---|
| N5・N6・N7b の途中 | 切れたセルから |
| N7 の途中 | N8b → N7(動いている印が残っているため) |
| N8 の途中 | N8b → N8 |
| N9 〜 N11 の途中 | N9 → N10a → N10b → N11(25_03 を回し直したことと、両方の回のまとめの数を `results/25_registration.txt` に書く) |
| N11b の後・N12b の前 | N9(h5 をローカルに作り直す)→ NR → N10a → N12 から(R1 は Drive にまだ無い。NR の sha256 に `_r1.pt` が出ない) |
| N12b の後・N15 の前 | N9(h5 をローカルに作り直す。N15 に要る)→ NR → N10a → 切れたセルから |
| N15 の途中 | N9 → NR → N10a → N15(24_07 を回し直したことを `results/25_registration.txt` に書く) |
| N15 の後・N17 の前 | N15 の最後の行で Drive に写せていれば 24_07 は回し直さず、NR → N10a → N17。N17 の値は、.pt を N12b の値と、CSV を JSON の `csv_sha256` と照らす。N16 の最後の行(seisbench)は `{LO}` を `{DRV}` に替えて回す。写す前に切れた時は「N15 の途中」と同じ |
| N17 の後 | N18 だけ(N2 の変数が要るので、N1 → N2 の後に) |